# Introduction to Agentic AI — ReAct 课堂作业（教师版）

本 Notebook 是教师演示与验收版本，包含三个 Reasoning 函数的参考实现。学生版为 `Introduction_ReAct_Learner.ipynb`。

## 教学目标

- 对比一次生成的 Direct 与可观察、可修正的 ReAct；
- 讲解 Few-shot 如何示范行动协议；
- 讲解 Reflection 如何使用 Verifier 的具体反馈；
- 说明解析器、工具、验证器和步数限制在 Harness 中的作用。

默认使用 `USE_REAL_API=False`，便于稳定演示和统一验收。


## 1. 任务背景：修正校园活动方案

学校计划举办150人的工作坊，需要轮椅通道和投影设备，持续2小时并在18:00前结束；讲者只能在14:00或16:00开始。总预算不超过1400元，外租投影仪费用为250元。

| 场地 | 容量 | 轮椅通道 | 内置投影 | 可用开始时间 | 场地费 |
|---|---:|:---:|:---:|---|---:|
| Hall A | 120 | 是 | 是 | 14:00、16:00 | 900 |
| Hall B | 180 | 是 | 否 | 14:00 | 1000 |
| Hall C | 160 | 否 | 是 | 16:00 | 800 |
| Hall D | 200 | 是 | 是 | 15:00 | 1300 |

初始方案：

```json
{"venue": "Hall C", "start": "16:00", "rent_projector": false}
```

Agent 可以使用 `VerifyPlan[JSON]`、`Calculate[expression]` 和 `Finish[JSON]`。


In [ ]:
import ast
import json
import operator
import os
import re
import urllib.error
import urllib.request
from getpass import getpass

MODEL = os.getenv("ZAI_MODEL", "glm-4-flash-250414")
USE_REAL_API = False      # False: deterministic offline run; True: call the Zhipu API
MAX_STEPS = 12

CONSTRAINTS = """150 attendees; wheelchair access required; projector required;
duration is 2 hours; finish by 18:00; presenter is available only at 14:00 or 16:00;
total cost must not exceed 1400.""".strip()

VENUES = {
    "Hall A": {"capacity": 120, "accessible": True,  "projector": True,  "slots": ["14:00", "16:00"], "fee": 900},
    "Hall B": {"capacity": 180, "accessible": True,  "projector": False, "slots": ["14:00"],          "fee": 1000},
    "Hall C": {"capacity": 160, "accessible": False, "projector": True,  "slots": ["16:00"],          "fee": 800},
    "Hall D": {"capacity": 200, "accessible": True,  "projector": True,  "slots": ["15:00"],          "fee": 1300},
}

PROJECTOR_RENTAL_FEE = 250
INITIAL_PLAN = {"venue": "Hall C", "start": "16:00", "rent_projector": False}
EXPECTED_PLAN = {"venue": "Hall B", "start": "14:00", "rent_projector": True, "total_cost": 1250}

TASK = f"""Repair the campus workshop plan. Constraints: {CONSTRAINTS}
Venue data: {json.dumps(VENUES)}
Initial plan: {json.dumps(INITIAL_PLAN)}
Projector rental fee: {PROJECTOR_RENTAL_FEE}.
Use VerifyPlan before Calculate. Finish only with venue, start, rent_projector, and total_cost."""

print("Model:", MODEL, "| Live API:", USE_REAL_API)


## 2. 运行模式

课堂演示建议保留 `USE_REAL_API=False`，不调用 API，且每次产生相同轨迹。需要展示真实模型差异时再切换为 `True`。

API Key 的申请、环境变量配置和安全注意事项见 [`README.md`](./README.md)。教师不应把共享 Key 写入或保存到 Notebook。


In [ ]:
class ZhipuClient:
    endpoint = "https://open.bigmodel.cn/api/paas/v4/chat/completions"

    def __init__(self, api_key=None):
        self.api_key = api_key or os.getenv("ZAI_API_KEY")
        if not self.api_key:
            print("未检测到 ZAI_API_KEY，请临时输入课堂 API Key。")
            self.api_key = getpass("智谱 API Key（输入不显示）: ")
        if not self.api_key:
            raise ValueError("Missing API key")

    def chat(self, messages, temperature=0.2, max_tokens=500):
        body = json.dumps({
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }).encode("utf-8")
        request = urllib.request.Request(
            self.endpoint,
            data=body,
            method="POST",
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"Zhipu API error {exc.code}: {detail}") from exc
        return payload["choices"][0]["message"]["content"]


class OfflineClient:
    """A deterministic client for classroom practice and grading."""

    def chat(self, messages, temperature=0.2, max_tokens=500):
        if "DIRECT_BASELINE" in messages[0]["content"]:
            return '{"venue":"Hall D","start":"15:00","rent_projector":false,"total_cost":1300}'

        task_index = max(i for i, message in enumerate(messages) if message.get("content") == TASK)
        active_messages = messages[task_index + 1:]
        observations = [
            message["content"] for message in active_messages
            if message["role"] == "user" and message["content"].startswith("Observation:")
        ]
        reflections = [
            message["content"] for message in active_messages
            if message["role"] == "user" and message["content"].startswith("Reflection:")
        ]

        if not active_messages:
            return ('Thought: I should verify the given plan before changing it.\n'
                    'Action: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]')
        if len(reflections) == 1 and not observations:
            return ('Thought: Hall D fixes access and capacity; verify it.\n'
                    'Action: VerifyPlan[{"venue":"Hall D","start":"15:00","rent_projector":false}]')
        if len(reflections) >= 2 and not observations:
            return ('Thought: Hall B at 14:00 meets timing; rent a projector and verify.\n'
                    'Action: VerifyPlan[{"venue":"Hall B","start":"14:00","rent_projector":true}]')
        if observations and "VALID" in observations[-1] and "1250" not in observations[-1]:
            return ('Thought: The plan is valid; calculate venue plus projector rental.\n'
                    'Action: Calculate[1000+250]')
        if observations and "Observation: 1250" in observations[-1]:
            return ('Thought: The valid plan and total cost are verified.\n'
                    'Action: Finish[{"venue":"Hall B","start":"14:00",'
                    '"rent_projector":true,"total_cost":1250}]')
        return ('Thought: I need to verify the initial plan.\n'
                'Action: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]')


client = ZhipuClient() if USE_REAL_API else OfflineClient()


## 3. Direct 基线

Direct 只有一次模型生成，没有工具反馈和第二次尝试。运行前可让学生判断：弱模型能否一次满足容量、无障碍、设备、时间和预算全部约束？


In [ ]:
DIRECT_SYSTEM = """DIRECT_BASELINE
Answer once. You have no calculator, code execution, tools, or second attempt.
Return the requested JSON and do not claim to have used a tool.""".strip()

direct_text = client.chat([
    {"role": "system", "content": DIRECT_SYSTEM},
    {"role": "user", "content": TASK},
])
print(direct_text)


## 4. 已提供的工具

`VerifyPlan` 检查容量、无障碍、设备、时间和预算，并返回具体违规项；`Calculate` 只负责最终费用。学生不需要实现工具。


In [ ]:
BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
}


def safe_calculate(expression):
    expression = expression.strip().strip("`").replace("×", "*").replace("÷", "/")

    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
            value = BIN_OPS[type(node.op)](visit(node.left), visit(node.right))
            if abs(value) > 10**15:
                raise ValueError("Intermediate result is too large")
            return value
        raise ValueError("Only numeric arithmetic is allowed")

    value = visit(ast.parse(expression, mode="eval"))
    return str(int(value) if isinstance(value, float) and value.is_integer() else value)


def verify_plan(plan):
    if not isinstance(plan, dict) or plan.get("venue") not in VENUES:
        return "INVALID | unknown venue or invalid JSON"

    venue = VENUES[plan["venue"]]
    start = str(plan.get("start", ""))
    rent_projector = plan.get("rent_projector") is True
    violations = []

    if venue["capacity"] < 150:
        violations.append("capacity below 150")
    if not venue["accessible"]:
        violations.append("wheelchair access required")
    if start not in venue["slots"]:
        violations.append("venue unavailable at selected time")
    if start not in ("14:00", "16:00"):
        violations.append("presenter unavailable at selected time")
    if start and start[:2].isdigit() and int(start[:2]) + 2 > 18:
        violations.append("event finishes after 18:00")
    if not venue["projector"] and not rent_projector:
        violations.append("projector required")

    cost = venue["fee"] + (PROJECTOR_RENTAL_FEE if rent_projector else 0)
    if cost > 1400:
        violations.append("budget exceeded")

    if violations:
        return "INVALID | " + "; ".join(violations)

    rental_fee = PROJECTOR_RENTAL_FEE if rent_projector else 0
    return f"VALID | venue_fee={venue['fee']} | projector_rental_fee={rental_fee}"


assert verify_plan(INITIAL_PLAN).startswith("INVALID")
assert verify_plan({"venue": "Hall B", "start": "14:00", "rent_projector": True}).startswith("VALID")
assert safe_calculate("1000+250") == "1250"
print("Verifier and calculator checks passed.")


## 5. 三个 Reasoning 函数（教师参考实现）

学生需要完成 System Prompt、Few-shot trajectory 和 Reflection 三个函数。下面是实测可用于 `glm-4-flash-250414` 的参考实现。

教学重点不是要求学生逐字复制，而是检查三个概念：完整 Action 协议、利用最新 Observation、根据失败字段进行针对性修正。


In [ ]:
def build_reasoning_prompt(constraints):
    return f"""You are a concise ReAct planning agent. Constraints: {constraints}
The user message contains authoritative venue data. Copy all numbers exactly; never invent facts.
First, verify the exact initial plan. Before every Action, write one short Thought based on the latest Observation.
Every VerifyPlan JSON MUST contain all three fields: venue, start, rent_projector.
After INVALID, revise every violated field and do not retry a venue with a permanent capacity or accessibility violation.
After VALID, use the observed venue_fee and projector_rental_fee in one Calculate Action.
After observing the calculated cost, Finish with exactly venue, start, rent_projector, total_cost.
Output exactly one Action per turn: VerifyPlan[JSON], Calculate[expression], or Finish[JSON]."""


def build_few_shot_messages():
    return [
        {"role": "user", "content": "DEMO ONLY: 60 people. Room X has 50 seats. Room Y has 80 seats, is available 10:00, has a projector, fee 600. Initial plan: Room X, 10:00, no rental."},
        {"role": "assistant", "content": 'Thought: I must verify the complete initial plan first.\nAction: VerifyPlan[{"venue":"Room X","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: INVALID | capacity below 60\nReflection: Room X permanently violates capacity; keep 10:00 and no rental, but change the venue."},
        {"role": "assistant", "content": 'Thought: Room Y fixes capacity while preserving the valid fields.\nAction: VerifyPlan[{"venue":"Room Y","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: VALID | venue_fee=600 | projector_rental_fee=0\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan is valid, so calculate the observed fees.\nAction: Calculate[600+0]'},
        {"role": "user", "content": "Observation: 600\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan and cost are now verified.\nAction: Finish[{"venue":"Room Y","start":"10:00","rent_projector":false,"total_cost":600}]'},
    ]


def build_reflection(feedback, previous_plan):
    return (
        f"Reflection: The previous plan {previous_plan} failed because: {feedback}. "
        "Preserve fields that satisfy the constraints, revise every violated field, and do not repeat "
        "a venue with a permanent capacity or accessibility violation. The next VerifyPlan must include "
        "venue, start, and rent_projector, followed by exactly one Action."
    )


## 6. ReAct + Reflection 框架

框架将 Few-shot 加入上下文，每轮解析并执行一个 Action，再将 Observation 返回模型。错误方案或错误 Finish 会触发 Reflection。


In [ ]:
ACTION_RE = re.compile(
    r"^\s*Action\s*:\s*(VerifyPlan|Calculate|Finish)\s*\[(.*?)\]\s*$",
    re.I | re.M | re.S,
)


def extract_plan(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and "venue" in value:
            return value

    for candidate in re.findall(r"\{[^{}]{1,300}\}", text, re.S):
        try:
            value = ast.literal_eval(candidate)
        except (SyntaxError, ValueError):
            continue
        if isinstance(value, dict) and "venue" in value:
            return value
    return None


def run_react(client, task, max_steps=12):
    system_prompt = build_reasoning_prompt(CONSTRAINTS)
    few_shot = build_few_shot_messages()

    if not isinstance(system_prompt, str) or not system_prompt.strip():
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_prompt_TODO"}
    if not isinstance(few_shot, list) or len(few_shot) < 6:
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_few_shot_TODO"}

    messages = [
        {"role": "system", "content": system_prompt},
        *few_shot,
        {"role": "user", "content": task},
    ]
    trace = []
    valid_plan = None
    calculated_cost = None
    previous_plan = INITIAL_PLAN

    for step in range(1, max_steps + 1):
        model_text = client.chat(messages)
        messages.append({"role": "assistant", "content": model_text})

        match = ACTION_RE.search(model_text)
        action = (match.group(1).title(), match.group(2).strip()) if match else None
        if action is None:
            bare_plan = extract_plan(model_text)
            action = ("Finish", model_text) if bare_plan else None

        if action is None:
            observation = "Format error: use exactly one VerifyPlan[...], Calculate[...], or Finish[...]."

        elif action[0].lower() == "verifyplan":
            plan = extract_plan(action[1])
            previous_plan = plan or previous_plan
            observation = verify_plan(plan)
            if observation.startswith("INVALID"):
                reflection = build_reflection(observation, previous_plan)
                if not isinstance(reflection, str) or not reflection.startswith("Reflection:"):
                    return {
                        "answer": None,
                        "trace": trace,
                        "passed": False,
                        "reason": "complete_reflection_TODO",
                    }
                trace.append({
                    "step": step,
                    "model": model_text,
                    "action": action,
                    "observation": observation,
                    "reflection": reflection,
                })
                messages.append({"role": "user", "content": reflection})
                continue
            valid_plan = plan

        elif action[0].lower() == "calculate":
            try:
                observation = safe_calculate(action[1])
            except (SyntaxError, ValueError, ZeroDivisionError, OverflowError) as exc:
                observation = f"Calculation error: {exc}"
            if observation.isdigit():
                calculated_cost = int(observation)

        else:
            answer = extract_plan(action[1]) or extract_plan(model_text)
            verified_core = {"venue": "Hall B", "start": "14:00", "rent_projector": True}
            process_ok = valid_plan == verified_core and calculated_cost == 1250
            answer_ok = answer == EXPECTED_PLAN
            if process_ok and answer_ok:
                trace.append({
                    "step": step,
                    "model": model_text,
                    "action": action,
                    "observation": None,
                })
                return {"answer": answer, "trace": trace, "passed": True, "reason": "finish"}

            feedback = (
                "Finish rejected: the plan must be verifier-approved and total_cost "
                "must equal the calculator result."
            )
            reflection = build_reflection(feedback, answer or previous_plan)
            if not isinstance(reflection, str) or not reflection.startswith("Reflection:"):
                return {
                    "answer": None,
                    "trace": trace,
                    "passed": False,
                    "reason": "complete_reflection_TODO",
                }
            trace.append({
                "step": step,
                "model": model_text,
                "action": action,
                "observation": feedback,
                "reflection": reflection,
            })
            messages.append({"role": "user", "content": reflection})
            continue

        trace.append({
            "step": step,
            "model": model_text,
            "action": action,
            "observation": observation,
        })
        messages.append({
            "role": "user",
            "content": f"Observation: {observation}\nContinue with exactly one Action.",
        })

    return {"answer": None, "trace": trace, "passed": False, "reason": "max_steps"}


In [ ]:
react_result = run_react(client, TASK, MAX_STEPS)

for item in react_result["trace"]:
    print(f"\n--- Step {item['step']} ---")
    print(item["model"])
    if item["observation"] is not None:
        print("Observation:", item["observation"])
    if item.get("reflection"):
        print(item["reflection"])

print("\nResult:", react_result)


## 7. 教师验收标准与答案

标准离线轨迹共5步：

1. 验证初始 Hall C 方案，因缺少轮椅通道被拒绝并触发 Reflection；
2. 验证 Hall D，因讲者15:00不可用被拒绝并触发 Reflection；
3. 验证 Hall B、14:00并租用投影仪，得到 `VALID`；
4. 使用 `Calculate[1000+250]` 得到1250；
5. 使用完整 JSON 执行 `Finish`，最终显示 `passed=True`。

唯一有效方案为：

```json
{
  "venue": "Hall B",
  "start": "14:00",
  "rent_projector": true,
  "total_cost": 1250
}
```

真实模型的候选方案和步数允许不同，但至少应根据一次 Verifier 反馈修改方案，并在 `MAX_STEPS` 内完成验证、计算和 Finish。


In [ ]:
direct_answer = extract_plan(direct_text)
print(f"Direct: answer={direct_answer!r}, pass={direct_answer == EXPECTED_PLAN}")
print(f"ReAct: answer={react_result['answer']!r}, pass={react_result['passed']}")

if react_result["passed"]:
    print("完成：Verifier 反馈、Few-shot 与 Reflection 共同驱动方案修正。")
else:
    print("尚未完成：请检查 Reasoning Prompt、Few-shot 和 Reflection 的实现。")
